# Lesson 04
Run lessons 1–3 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第4课 替代数据与金融NLP

交付：真实10-K/10-Q文档清单、SEC acceptance/accession/报告期/版本映射、全文提取与段落质量检查、LM情绪及不确定性、否定处理、五交易日超额方向标签、事前市场模型事件研究、价格/价格+词典/价格+TF-IDF Logistic 对照和增量评价。

SEC全文端点本次在Colab返回403，遵守访问限制；使用公司官网列出的IBM原始申报PDF镜像，不改变申报元数据。以报告期、表单、CIK匹配SEC accession；缺失、重复链接、报告期不符、版本歧义均记录并剔除。保存完整下载字节哈希；主文件包含的引用外部附件不自动假定已经下载。镜像历史内容可能更新，无法保证与当年SEC字节完全相同，这是实际PIT局限。

LM官方[词典](https://sraf.nd.edu/loughranmcdonald-master-dictionary/)用于学术教学；官方CSV/XLSX因下载配额不可用时，使用[pysentiment2 0.1.1随附的LM.csv快照](https://nickderobertis.github.io/pysentiment/_modules/pysentiment2/lm.html)，显示实际来源与哈希，不冒充当前官网版本。保存实际词典版本，主实验冻结2011年已存在的情绪词标记，仅用基础词源，不使用2025词频。该重建不是完整2011字典快照，保留修订偏差。TF-IDF词表、数值填补和标准化均只fit训练集。FinBERT仅列为冻结推断扩展，不微调大模型。

时间链：保留acceptance原字段；执行下界取精确时间与filed日纽约时间23:59较晚者→加60分钟处理延迟→下一个可交易开盘→第五个后续开盘退出。即使有精确acceptance也采用保守filed日末下界；旧元数据缺失精确时间时只使用filed日末，并保留原字段缺失状态。120个已经结束的开盘到开盘收益估计市场模型，未来五日仅构造标签；同一报告组不跨集合，切分边界清除未成熟标签。

IBM单公司2014—2022小样本仅用于流程、数据质量和增量诊断，不能声称已经证明文本有交易价值。训练≤2018、验证2019—2020、开发评估2021—2022；不随机打乱，不根据最后一段挑词、挑事件或挑模型。事件研究用市场模型异常收益；分类标签使用股票与SPY同窗复合收益之差，两者定义分别输出。

参考：[SEC API](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、[IBM原始报告镜像目录](https://www.ibm.com/investor/services/investor-information)、[sklearn Pipeline防泄漏](https://scikit-learn.org/stable/common_pitfalls.html)。


In [ ]:
# 04.01 真实申报元数据、公司官网镜像映射与LM词典
from pypdf import PdfReader
IBM_META=json.loads(fetch('https://data.sec.gov/submissions/CIK0000051143.json'))
parts=[pd.DataFrame(IBM_META['filings']['recent'])]
for file in IBM_META['filings'].get('files',[]):
    if file['filingTo']>='2014-01-01' and file['filingFrom']<='2022-12-31':
        parts.append(pd.DataFrame(json.loads(fetch('https://data.sec.gov/submissions/'+file['name']))))
FILINGS=pd.concat(parts,ignore_index=True).drop_duplicates('accessionNumber')
FILINGS=FILINGS[FILINGS.form.isin(['10-K','10-Q','10-K/A','10-Q/A'])].copy()
FILINGS['reportDate']=pd.to_datetime(FILINGS.reportDate); FILINGS['filingDate']=pd.to_datetime(FILINGS.filingDate)
FILINGS['acceptance_raw']=FILINGS.get('acceptanceDateTime',pd.Series(index=FILINGS.index,dtype=str))
FILINGS['acceptance_at']=pd.to_datetime(FILINGS.acceptance_raw,utc=True,errors='coerce').dt.tz_convert(NY)
FILINGS['filed_end_at']=(FILINGS.filingDate+pd.Timedelta(hours=23,minutes=59)).dt.tz_localize(NY)
# 保守事件版本：有精确时间也不早于filed日末；避免旧API时区或当天处理能力的乐观假设。
FILINGS['public_bound']=FILINGS[['acceptance_at','filed_end_at']].max(axis=1)
FILINGS['ready_at']=FILINGS.public_bound+pd.Timedelta(hours=1)
FILINGS=FILINGS[FILINGS.filingDate.between('2014-01-01','2022-12-31')]
LM_XLSX_URL='https://drive.google.com/uc?export=download&id=1P3KtWuDyQT2Eu_VwTU3D2sEeQxR9dWCS'
subprocess.check_call([sys.executable,'-m','pip','install','-q','openpyxl==3.1.5'])
try:
    LM=pd.read_excel(io.BytesIO(fetch(LM_XLSX_URL)))
    LM_SOURCE='official 1993-2025 workbook'
    LM_BLOB=fetch(LM_XLSX_URL)
except Exception as e:
    print('Official workbook unavailable:',str(e)); print('Use declared pinned package snapshot; do not claim latest official file')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps','pysentiment2==0.1.1'])
    LM_BLOB=im.distribution('pysentiment2').locate_file('pysentiment2/static/LM.csv').read_bytes()
    LM=pd.read_csv(io.BytesIO(LM_BLOB)); LM_SOURCE='pysentiment2 0.1.1 bundled LM.csv (package released 2020)'
print('Dictionary source',LM_SOURCE,'sha256',hashlib.sha256(LM_BLOB).hexdigest())
assert {'Word','Negative','Positive','Uncertainty','Source'}.issubset(LM.columns) and len(LM)>10000,'Official LM schema failed'
LM.columns=LM.columns.str.strip(); LM['Word']=LM.Word.astype(str).str.upper()
base=LM[LM.Source.astype(str).eq('12of12inf')]
# 只取当时已有的词类标签，正负年标记是加入/移除年，不把negative年当情绪负值。
LEX={c:set(base.loc[base[c].between(1,2011),'Word']) for c in ['Positive','Negative','Uncertainty']}
EFFECTIVE_VOCAB=set(base.Word); NEGATORS={'NO','NOT','NEVER','NEITHER','NOR','WITHOUT','CANNOT'}
print('LM snapshot rows',len(LM),'frozen vocab',len(EFFECTIVE_VOCAB),'categories',{k:len(v) for k,v in LEX.items()})
# 来源仅在官网可见链接中选择；没有精确映射的链接不会进入实验。
LINK_MANIFEST=[]
for link in IBM_LINKS:
    url=link['url']; label=link['label']; name=url.rsplit('/',1)[-1]
    year_match=re.search(r'(20\d{2})',name)
    year=int(year_match[1]) if year_match else (2000+int(re.search(r'^[123]q(\d{2})_',name,re.I)[1]) if re.search(r'^[123]q(\d{2})_',name,re.I) else None)
    form='10-K' if label=='10-K' else '10-Q'
    qm=re.search(r'([123])(?:st|nd|rd) Quarter',label); q=int(qm[1]) if qm else 4
    if year is None or not 2013<=year<=2022: continue
    report_end=pd.Timestamp(year=year,month=3*q,day=1)+pd.offsets.MonthEnd(0)
    m=FILINGS[(FILINGS.form==form)&(FILINGS.reportDate==report_end)]
    status='mapped' if len(m)==1 else 'missing_or_ambiguous'
    record={'ticker':'IBM','CIK':51143,'form':form,'report_end':report_end,'url':url,'status':status,'directory_label':label}
    if len(m)==1: record.update(m.iloc[0][['accessionNumber','acceptance_raw','acceptance_at','filed_end_at','ready_at','filingDate']].to_dict())
    LINK_MANIFEST.append(record)
LINK_MANIFEST=pd.DataFrame(LINK_MANIFEST)
LINK_MANIFEST.loc[LINK_MANIFEST.url.duplicated(keep=False),'status']='duplicate_directory_url'
print('Amendments kept in separate audit, never merged into original event'); display(FILINGS[FILINGS.form.str.endswith('/A')][['form','reportDate','accessionNumber','filingDate']])
display(LINK_MANIFEST[['form','report_end','status','accessionNumber','url']]); print('04.01 manifest prepared')


In [ ]:
# 04.02 原始PDF全文提取、质量筛选、段落与否定词典特征
TEXTS={}; DOC_ROWS=[]; EXTRACT_CACHE=globals().get('EXTRACT_CACHE',{})
def text_features(text):
    seq=re.findall(r'[A-Za-z]+|[.!?;:]',text.upper()); counts=dict(positive=0,negative=0,uncertainty=0,negated=0); history=[]; effective=0
    for token in seq:
        if token in '.!?;:': history=[]; continue
        effective+=int(token in EFFECTIVE_VOCAB)
        neg=any(x in NEGATORS for x in history[-3:])
        if token in LEX['Positive']:
            counts['negative' if neg else 'positive']+=1; counts['negated']+=int(neg)
        if token in LEX['Negative']:
            if not neg: counts['negative']+=1
            else: counts['negated']+=1
        if token in LEX['Uncertainty']: counts['uncertainty']+=1
        history.append(token)
    den=max(effective,1)
    return {'sentiment':(counts['positive']-counts['negative'])/den,'negative':counts['negative']/den,
            'uncertainty':counts['uncertainty']/den,'negated_rate':counts['negated']/den,'effective_tokens':effective,'log_length':np.log1p(effective)}
for _,r in LINK_MANIFEST.iterrows():
    row=r.to_dict(); row['text_scope']='complete downloaded main filing PDF; external incorporated exhibits not collected'
    if r.status!='mapped': DOC_ROWS.append(row); continue
    try:
        blob=fetch(r.url); assert blob.startswith(b'%PDF'),'not a PDF'
        blob_hash=hashlib.sha256(blob).hexdigest()
        if blob_hash in EXTRACT_CACHE: pages=EXTRACT_CACHE[blob_hash]
        else:
            reader=PdfReader(io.BytesIO(blob)); pages=[page.extract_text() or '' for page in reader.pages]; EXTRACT_CACHE[blob_hash]=pages
        full='\n\n'.join(pages); header=re.sub(r'\s+',' ',full[:7000]).upper()
        expected=r.report_end.strftime('%B').upper()+r.report_end.strftime(' %d, %Y')
        # 不仅依赖官网链接标签：验证文本中的表单、年份、报告期和发行人。
        dates_found=[pd.to_datetime(m.group(0)).normalize() for m in re.finditer(r'(?:MARCH|JUNE|SEPTEMBER|DECEMBER)\s+\d{1,2},?\s+20\d{2}',header)]
        assert r.report_end in dates_found,'PDF report period mismatch'
        assert 'INTERNATIONAL BUSINESS MACHINES' in header,'issuer mismatch'
        assert 'FORM10'+r.form[-1] in re.sub(r'[^A-Z0-9]','',header),'form mismatch'
        # 保留完整原文；清洗只规范空白，不混入仅部分摘要。段落去重仅用于QA统计。
        paragraphs=[re.sub(r'\s+',' ',p).strip() for p in re.split(r'\n\s*\n',full) if len(p.strip())>40]
        features=text_features(full)
        assert features['effective_tokens']>1000,'extraction too short'
        row.update(features); row.update(status='usable',pages=len(pages),chars=len(full),paragraphs=len(paragraphs),
            repeated_paragraphs=len(paragraphs)-len(set(paragraphs)),sha256=hashlib.sha256(blob).hexdigest(),
            text_sha256=hashlib.sha256(full.encode()).hexdigest(),event_group='IBM_'+str(r.report_end.date())+'_'+r.form)
        TEXTS[r.accessionNumber]=full
    except Exception as e: row.update(status='rejected',reason=str(e))
    DOC_ROWS.append(row)
DOCS=pd.DataFrame(DOC_ROWS); EVENTS=DOCS[DOCS.status.eq('usable')].copy().sort_values('ready_at')
print(DOCS[['report_end','form','status']].to_string(index=False)); print(DOCS.get('reason',pd.Series(dtype=str)).value_counts().to_dict())
assert len(EVENTS)>=20,'Insufficient real documents: stop and inspect manifest'
assert EVENTS.accessionNumber.is_unique and EVENTS.event_group.is_unique
EVENTS['text']=EVENTS.accessionNumber.map(TEXTS)
print('真实全文质量报告'); display(DOCS[['form','report_end','status','pages','chars','effective_tokens','sha256','url']])
print('有效事件',len(EVENTS),'form counts',EVENTS.form.value_counts().to_dict(),'字节与文本均仅内存缓存')
print('规则：3词否定窗口；否定正面词转负面，否定负面词忽略；遇标点重置；这只是透明基线。')
pos=sorted(LEX['Positive'])[0]; neg=sorted(LEX['Negative'])[0]
assert text_features(pos)['sentiment']>text_features('NOT '+pos)['sentiment']
assert text_features(neg)['negative']>text_features('NOT '+neg)['negative']
print('04.02 PASS: actual PDF text and negation unit checks')


In [ ]:
# 04.03 公告事件时间轴、五日标签和事前市场模型
OPEN_TIMES=(OPEN.index+pd.Timedelta(hours=9,minutes=30)).tz_localize(NY)
CLOSE_TIMES=(CLOSE.index+pd.Timedelta(hours=16)).tz_localize(NY)
OO=OPEN[['IBM','SPY']].shift(-1)/OPEN[['IBM','SPY']]-1
END_TIMES=pd.Series(OPEN_TIMES,index=OPEN.index).shift(-1)
EVENT_ROWS=[]; AR_ROWS=[]
for _,r in EVENTS.iterrows():
    j=OPEN_TIMES.searchsorted(r.ready_at,side='left')
    if j<120 or j+5>=len(OPEN): continue
    entry=OPEN.index[j]; exit_session=OPEN.index[j+5]
    known=CLOSE_TIMES.searchsorted(r.ready_at,side='right')-1
    hist=CLOSE.iloc[:known+1]
    fit_cutoff=r.filingDate.tz_localize(NY)
    if pd.notna(r.acceptance_at): fit_cutoff=min(fit_cutoff,r.acceptance_at)
    eligible=OO.loc[(END_TIMES<=fit_cutoff).fillna(False)].dropna().tail(120)
    if len(eligible)<120: continue
    model=sm.OLS(eligible.IBM,sm.add_constant(eligible.SPY)).fit()
    future=OO.iloc[j:j+5]
    abnormal=future.IBM-(model.params['const']+model.params['SPY']*future.SPY)
    for k,ar in enumerate(abnormal): AR_ROWS.append({'accessionNumber':r.accessionNumber,'event_day':k+1,'AR':ar})
    rx=OPEN.IBM.iloc[j+5]/OPEN.IBM.iloc[j]-1; rm=OPEN.SPY.iloc[j+5]/OPEN.SPY.iloc[j]-1
    row=r.to_dict(); row.update(entry_session=entry,entry_at=OPEN_TIMES[j],label_end=OPEN_TIMES[j+5],
      future_excess=rx-rm,y=int(rx-rm>0),CAR5=abnormal.sum(),alpha_pre=model.params['const'],beta_pre=model.params['SPY'],
      fit_cutoff=fit_cutoff,fit_last_end=END_TIMES.loc[eligible.index[-1]],feature_last_close=CLOSE_TIMES[known],
      price_r5=hist.IBM.iloc[-1]/hist.IBM.iloc[-6]-1,price_r20=hist.IBM.iloc[-1]/hist.IBM.iloc[-21]-1,
      price_vol20=hist.IBM.pct_change(fill_method=None).tail(20).std(),market_r20=hist.SPY.iloc[-1]/hist.SPY.iloc[-21]-1)
    EVENT_ROWS.append(row)
E4=pd.DataFrame(EVENT_ROWS).sort_values('entry_at').reset_index(drop=True)
E4['fold']=np.select([E4.entry_session<TRAIN_END,E4.entry_session<VALID_END],['train','validation'],default='development_eval')
train_boundary=pd.Timestamp(TRAIN_END).tz_localize(NY); valid_boundary=pd.Timestamp(VALID_END).tz_localize(NY)
E4['purged']=((E4.fold=='train')&(E4.label_end>=train_boundary))|((E4.fold=='validation')&(E4.label_end>=valid_boundary))
E4=E4[~E4.purged].copy().reset_index(drop=True)
assert (E4.entry_at>=E4.ready_at).all() and (E4.feature_last_close<E4.entry_at).all()
assert (E4.fit_last_end<=E4.fit_cutoff).all()
assert E4.groupby('event_group').fold.nunique().max()==1
print('事件时间轴：精确申报时间保留；执行使用filed日末保守下界加一小时。')
display(E4[['accessionNumber','form','report_end','acceptance_raw','ready_at','entry_at','label_end','fit_cutoff','fit_last_end','fold','y','future_excess','CAR5']])
display(E4.groupby('fold').agg(n=('y','size'),positive=('y','sum'),mean_excess=('future_excess','mean')))
AR=pd.DataFrame(AR_ROWS).merge(E4[['accessionNumber']],on='accessionNumber',how='inner')
summary=AR.groupby('event_day').AR.agg(['mean','std','count']); summary['CAAR']=summary['mean'].cumsum()
fig,ax=plt.subplots(1,2,figsize=(12,4)); summary['CAAR'].plot(marker='o',ax=ax[0],title='Market-model mean cumulative abnormal return')
E4.plot.scatter(x='sentiment',y='future_excess',ax=ax[1],title='Dictionary sentiment vs 5-session excess'); plt.tight_layout(); plt.show()
print('04.03 PASS: observed events, 120 past market-model observations, 5 open-to-open steps')


In [ ]:
# 04.04 同窗价格、LM与TF-IDF Logistic增量对照
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import balanced_accuracy_score,roc_auc_score,brier_score_loss,log_loss,mean_squared_error
PRICE_FEATURES=['price_r5','price_r20','price_vol20','market_r20']
DICT_FEATURES=['sentiment','negative','uncertainty','negated_rate','log_length']
L4_EXPERIMENTS=pd.DataFrame([
 ['price_only',PRICE_FEATURES,False],['price+LM',PRICE_FEATURES+DICT_FEATURES,False],
 ['price+TFIDF',PRICE_FEATURES,True],['length_only',['log_length'],False]],columns=['model','numeric_features','TFIDF'])
def classifier(nums,text=False):
    num=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())])
    steps=[('num',num,nums)]
    if text: steps.append(('text',TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=.95,max_features=800,sublinear_tf=True),'text'))
    return Pipeline([('features',ColumnTransformer(steps)),('model',LogisticRegression(C=1,solver='liblinear',random_state=SEED,max_iter=2000))])
tr=E4[E4.fold.eq('train')]; va=E4[E4.fold.eq('validation')]; te=E4[E4.fold.eq('development_eval')]
assert len(tr)>=10 and tr.y.nunique()==2 and len(va)>=3 and len(te)>=3
L4_MODELS={}; L4_PRED=E4[['accessionNumber','event_group','entry_session','entry_at','label_end','fold','y','future_excess']].copy(); result=[]
for _,r in L4_EXPERIMENTS.iterrows():
    model=classifier(r.numeric_features,r.TFIDF); model.fit(tr,tr.y); L4_MODELS[r.model]=model
    L4_PRED[r.model]=model.predict_proba(E4)[:,1]
    for fold,data in [('validation',va),('development_eval',te)]:
        p=model.predict_proba(data)[:,1]
        result.append({'model':r.model,'fold':fold,'n':len(data),'balanced_accuracy':balanced_accuracy_score(data.y,p>=.5),
         'AUC':roc_auc_score(data.y,p) if data.y.nunique()==2 else np.nan,'Brier':brier_score_loss(data.y,p),'log_loss':log_loss(data.y,p,labels=[0,1])})
L4_METRICS=pd.DataFrame(result).set_index(['fold','model']); display(L4_EXPERIMENTS); display(L4_METRICS)
# 同一事件的配对误差差异；按年展示，避免把小样本差异写成统计显著结论。
paired=L4_PRED[L4_PRED.fold.ne('train')].copy(); paired['year']=paired.entry_session.dt.year
for model in ['price+LM','price+TFIDF']:
    paired['delta_Brier_'+model]=(paired[model]-paired.y)**2-(paired.price_only-paired.y)**2
DELTA=paired.groupby('year')[[c for c in paired if c.startswith('delta_Brier')]].mean(); display(DELTA)
print('delta<0代表该年Brier改善；小样本、镜像版本、词典重建及多次实验限制均保留。')
for model_name in ['price+LM','price+TFIDF']:
    base_metric=L4_METRICS.xs('price_only',level='model')['Brier']
    delta=L4_METRICS.xs(model_name,level='model')['Brier']-base_metric
    verdict='两个开发窗口Brier均降低，但小样本与版本局限下仍不足以证明稳定可交易增量' if (delta<0).all() else 'Brier改善没有在两个开发窗口一致出现，当前实验未支持稳定增量'
    print(model_name,'相对仅价格的Brier差',delta.to_dict(),'；结论：'+verdict)

# 字典密度只做有效词分母；TF-IDF不用全样本词频。运行验证集不会改变已fit词表。
v=L4_MODELS['price+TFIDF'].named_steps['features'].named_transformers_['text']; before_vocab=dict(v.vocabulary_)
v.transform(va.text); assert v.vocabulary_==before_vocab
fig,ax=plt.subplots(1,2,figsize=(12,4))
for model in ['price_only','price+LM','price+TFIDF']:
    ax[0].plot(paired.entry_session,paired[model],marker='o',label=model)
    bins=pd.cut(paired[model],[0,.25,.5,.75,1],include_lowest=True)
    cal=paired.groupby(bins,observed=True).agg(prob=(model,'mean'),frequency=('y','mean'),n=('y','size'))
    ax[1].plot(cal.prob,cal.frequency,marker='o',label=model)
ax[0].set_title('Chronological event predictions'); ax[0].legend(); ax[1].plot([0,1],[0,1],'k--'); ax[1].set_title('Small-sample calibration diagnostic'); ax[1].legend(); plt.tight_layout(); plt.show()
print('04.04 PASS: numeric and vocab fit only train; no event/random-row split')


In [ ]:
# 04.05 相同事件窗口的成本回测、模型卡与完整验收
L4_TARGETS={}; L4_LEDGERS={}; L4_TRADES={}
l4_dates=OPEN.loc['2019-01-01':'2022-12-29'].index
for name in L4_MODELS:
    w=pd.DataFrame(0.,index=l4_dates,columns=['IBM'])
    for _,r in L4_PRED[L4_PRED.fold.ne('train')].iterrows():
        j=OPEN.index.get_loc(r.entry_session); window=OPEN.index[j:j+5].intersection(l4_dates)
        w.loc[window,'IBM']=.8 if r[name]>=.5 else 0.
    L4_TARGETS[name]=w
    L4_LEDGERS[name],L4_TRADES[name]=l2_simulate_multi(OPEN[['IBM']],w,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
L4_ACCOUNT=pd.DataFrame({name:ledger_stats(x) for name,x in L4_LEDGERS.items()}).T; display(L4_ACCOUNT)
L4_CHECKS=[]
def check4(name,ok):
    assert bool(ok),name
    L4_CHECKS.append({'check':name,'status':'PASS'})
check4('actual docs with hash',EVENTS.sha256.str.len().eq(64).all() and len(EVENTS)>=20)
check4('unique accession + event',E4.accessionNumber.is_unique and E4.event_group.is_unique)
check4('document first-public conservative bound', (E4.ready_at>=E4.filed_end_at+pd.Timedelta(hours=1)).all())
check4('entry after readiness',(E4.entry_at>=E4.ready_at).all())
check4('market model uses past ended returns',(E4.fit_last_end<E4.ready_at).all())
check4('market model excludes announcement-day returns',(E4.fit_last_end<=E4.fit_cutoff).all())
check4('price feature availability',(E4.feature_last_close<E4.entry_at).all())
check4('5 trading-session exits',all(OPEN.index.get_loc(x.label_end.tz_localize(None).normalize())-OPEN.index.get_loc(x.entry_session)==5 for _,x in E4.iterrows()))
check4('event group folds',E4.groupby('event_group').fold.nunique().max()==1)
check4('train labels purged',tr.label_end.max()<train_boundary)
check4('validation labels purged',va.label_end.max()<valid_boundary)
check4('TFIDF train vocab frozen',v.vocabulary_==before_vocab)
for name,x in L4_LEDGERS.items():
    check4(name+' common window',x.index.equals(l4_dates))
    check4(name+' accounting',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
    check4(name+' gross long cap',x.gross_exposure.max()<=.800001 and (L4_TARGETS[name]>=0).all().all())
L4_VALIDATION=pd.DataFrame(L4_CHECKS); display(L4_VALIDATION)
L4_CARD={'course':4,'status':'verified','documents':len(EVENTS),'labeled_events':len(E4),
 'forms':EVENTS.form.value_counts().to_dict(),'dictionary_source':LM_SOURCE,'dictionary_sha256':hashlib.sha256(LM_BLOB).hexdigest(),
 'train_only_fit':['numeric median','standard scaler','TFIDF vocabulary/IDF','Logistic'],
 'checks':len(L4_CHECKS),'limitations':['single surviving issuer','small samples','official SEC archive 403; issuer main-filing mirrors','conservative filed-end timing; acceptance preserved','incorporated exhibits not collected','historical lexicon reconstructed from packaged snapshot, not exact 2011 release','no causal incremental-value claim','development data already observed']}
fig,ax=plt.subplots(figsize=(12,4)); pd.DataFrame({n:x.wealth for n,x in L4_LEDGERS.items()}).plot(ax=ax,title='Same-event 5-session signals with 5bp cost per traded leg'); plt.tight_layout(); plt.show()
print(json.dumps(L4_CARD,ensure_ascii=False,indent=2)); print('L4 FINAL PASS')
